# Self-RAG Generative Answer Verifier

Side experiment: fine-tune **Flan-T5** to generate Self-RAG reflection tokens
(`[IsRel]`, `[IsSup]`, `[IsUse]`, `[Decision]`) instead of the thesis-core
DeBERTa classification head.

- **Input:** `(question, gold_context, candidate_answer)`
- **Output:** reflection string ending in `ACCEPT` or `REJECT`
- **No retriever** — gold context from `labeled_asqa.csv`

Logic lives in `experiments/self_rag_verifier/verifier_system.py`.
Config: `configs/experiments/rag_verifier.yaml`.

## 1. Setup

In [ ]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from src.rag_filtering.config.loader import load_yaml, resolve_path
from src.self_rag_verifier.verifier_system import VerifierSystem

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

CONFIG_PATH = "configs/experiments/rag_verifier.yaml"
SPLIT = "test"

# Set RUN_TRAIN=True to fine-tune (slow on CPU; use GPU/Kaggle for full runs).
RUN_TRAIN = False
RESUME_TRAINING = False

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print(f"Project root: {PROJECT_ROOT}")

## 2. Config

In [ ]:
cfg = load_yaml(CONFIG_PATH)
print("Model:", cfg["model"]["name"])
print("Checkpoint:", cfg["model"]["model_save_dir"])
print("Results:", cfg["evaluation"]["results_dir"])
print("Epochs:", cfg["training"]["num_epochs"], "| batch:", cfg["training"]["batch_size"])
print("\nReflection targets:")
print("  accept:", cfg["reflection_tokens"]["accept_target"])
print("  reject:", cfg["reflection_tokens"]["reject_target"])

## 3. Load Data

In [ ]:
verifier = VerifierSystem(config_path=CONFIG_PATH)
verifier.load_data()

print(f"Train: {len(verifier.train_df)}, Val: {len(verifier.val_df)}, Test: {len(verifier.test_df)}")
print("Label balance (train):", verifier.train_df["label"].value_counts().to_dict())

## 4. Prompt and Target Preview

In [ ]:
sample_pos = verifier.train_df[verifier.train_df["label"] == 1].iloc[0]
sample_neg = verifier.train_df[verifier.train_df["label"] == 0].iloc[0]

for label, row in [(1, sample_pos), (0, sample_neg)]:
    ctx = verifier._extract_context(str(row["context"]))
    prompt = verifier._build_prompt(str(row["question"]), ctx, str(row["answer"]))
    target = verifier._build_target(int(row["label"]))
    print(f"\n=== label={label} ===")
    print("PROMPT:", prompt[:400], "..." if len(prompt) > 400 else "")
    print("TARGET:", target)

## 5. Train or Load Checkpoint

Set `RUN_TRAIN = True` to fine-tune Flan-T5. Otherwise the notebook loads
`models/answer_verifier/` if it exists.

> **Note:** Checkpoint dir must be `models/answer_verifier/` — paths containing
> the substring `rag` break Transformers heuristics on empty directories.

In [ ]:
checkpoint_dir = resolve_path(cfg["model"]["model_save_dir"])

if RUN_TRAIN:
    save_path = verifier.train(resume_from_checkpoint=RESUME_TRAINING)
    print(f"Training complete. Saved to {save_path}")
else:
    verifier.load_model()
    print(f"Loaded checkpoint from {checkpoint_dir}")

## 6. Evaluate

In [ ]:
metrics = verifier.evaluate(split=SPLIT)

summary = {
    "split": metrics["split"],
    "n_samples": metrics["n_samples"],
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "fpr": metrics["fpr"],
    "tp": metrics["tp"],
    "tn": metrics["tn"],
    "fp": metrics["fp"],
    "fn": metrics["fn"],
}
print(json.dumps(summary, indent=2))

results_path = resolve_path(cfg["evaluation"]["results_dir"]) / f"metrics_{SPLIT}.json"
print(f"\nMetrics saved to {results_path}")

In [ ]:
y_true = [0] * metrics["tn"] + [0] * metrics["fp"] + [1] * metrics["fn"] + [1] * metrics["tp"]
y_pred = [0] * metrics["tn"] + [1] * metrics["fp"] + [0] * metrics["fn"] + [1] * metrics["tp"]
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots()
ConfusionMatrixDisplay(
    cm,
    display_labels=["REJECT (0)", "ACCEPT (1)"],
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Self-RAG Verifier — {SPLIT} split")
plt.tight_layout()
plt.show()

## 7. Sample Predictions

In [ ]:
eval_df = verifier.test_df if SPLIT == "test" else (
    verifier.val_df if SPLIT == "val" else verifier.train_df
)

preview = eval_df.sample(n=min(5, len(eval_df)), random_state=cfg["data"]["seed"])
rows = []

for _, row in preview.iterrows():
    pred = verifier.predict(
        str(row["question"]),
        str(row["context"]),
        str(row["answer"]),
    )
    rows.append({
        "label": int(row["label"]),
        "decision": pred["decision"],
        "correct": int(row["label"]) == (1 if pred["accept"] else 0),
        "raw_output": pred["raw_output"],
        "question": str(row["question"])[:80],
        "answer": str(row["answer"])[:80],
    })

pd.DataFrame(rows)

## 8. Error Spot-Check (False Positives / False Negatives)

In [ ]:
batch_size = cfg["training"]["batch_size"]
predictions = verifier.predict_batch(
    questions=eval_df["question"].astype(str).tolist(),
    contexts=eval_df["context"].astype(str).tolist(),
    answers=eval_df["answer"].astype(str).tolist(),
    batch_size=batch_size,
)

eval_df = eval_df.copy()
eval_df["pred_accept"] = [p["accept"] for p in predictions]
eval_df["raw_output"] = [p["raw_output"] for p in predictions]

false_positives = eval_df[(eval_df["label"] == 0) & (eval_df["pred_accept"])].head(3)
false_negatives = eval_df[(eval_df["label"] == 1) & (~eval_df["pred_accept"])].head(3)

print(f"False positives: {len(eval_df[(eval_df['label'] == 0) & (eval_df['pred_accept'])])}")
print(f"False negatives: {len(eval_df[(eval_df['label'] == 1) & (~eval_df['pred_accept'])])}")

for title, subset in [("FALSE POSITIVE (accepted bad answer)", false_positives),
                      ("FALSE NEGATIVE (rejected good answer)", false_negatives)]:
    print(f"\n=== {title} ===")
    for _, row in subset.iterrows():
        print(f"Q: {str(row['question'])[:100]}")
        print(f"A: {str(row['answer'])[:120]}")
        print(f"Generated: {row['raw_output']}\n")